# 04 — ABC vs Protocol : héritage nominal vs typage structurel

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre la différence entre **héritage nominal** (ABC) et **typage structurel** (Protocol) ;
- utiliser `abc.ABC` et `@abstractmethod` pour poser un contrat strict ;
- utiliser `typing.Protocol` pour exprimer un duck-type vérifiable statiquement ;
- utiliser `@runtime_checkable` et connaître ses limites ;
- savoir choisir le bon outil selon le contexte (API publique vs interne, bibliothèque vs application).


## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- le modèle objet complet (héritage, MRO, `super`, méthodes spéciales) ;
- les type hints modernes (`int | None`, génériques, `Protocol`, `TypeVar`) ;
- les dataclasses (`@dataclass`, `field`, `frozen=True`, `slots=True`) ;
- les décorateurs de fonction et de classe, et les gestionnaires de contexte ;
- les tests avec `pytest` (fixtures, paramétrage, monkeypatch) ;
- le packaging avec `pyproject.toml` et `uv` ;
- le logging (module `logging`, handlers, formatters) ;
- les bases de SQL et `sqlite3`, les expressions régulières.
- métaclasses et `__init_subclass__` (notebooks précédents).

Notions que nous allons **introduire ou approfondir** ici :

- le duck typing **statique** introduit par PEP 544 (`Protocol`) ;
- `@runtime_checkable` et ses pièges ;
- l'arbre de décision ABC vs Protocol.


## Plan

1. Rappel : duck typing dynamique
2. ABC — héritage nominal
3. Protocol — typage structurel
4. `@runtime_checkable` : vérification à l'exécution
5. ABC + Protocol combinés
6. Cas d'usage de chaque outil
7. Arbre de décision
8. Synthèse
9. Exercices


---

## 1. Rappel — duck typing dynamique

En Python, *if it walks like a duck and quacks like a duck, it's a duck*. Une fonction qui attend « un objet qui a une méthode `read()` » ne vérifie en général rien : si l'objet n'a pas la méthode, ça plantera à l'exécution.

Deux limites de cette approche :

1. **L'outillage ne peut rien dire** (mypy, IDE) : il ne sait pas ce qu'attend vraiment la fonction.
2. Les erreurs arrivent à l'exécution, loin du point d'erreur réel.

Deux solutions : les **ABC** (héritage explicite) et les **Protocols** (typage structurel).

In [ ]:
def lire_tout(source) -> str:
    return source.read()

# Ça marche avec n'importe quoi ayant .read() — y compris des objets
# qui n'ont aucun rapport sémantique :
class Fake:
    def read(self): return "fake"

print(lire_tout(Fake()))

---

## 2. ABC — héritage nominal

Une **Abstract Base Class** est une classe qui définit un contrat par **héritage explicite** : pour être considérée comme une `Stream`, votre classe doit **hériter** de `Stream`.


In [ ]:
from abc import ABC, abstractmethod

class Stream(ABC):
    @abstractmethod
    def read(self) -> str: ...

    @abstractmethod
    def close(self) -> None: ...

    def __enter__(self): return self
    def __exit__(self, *exc): self.close()

class StringStream(Stream):
    def __init__(self, s: str) -> None:
        self._s = s
    def read(self) -> str:
        return self._s
    def close(self) -> None:
        self._s = ""

with StringStream("bonjour") as s:
    print(s.read())

Caractéristiques :

- **Contrat explicite** : `StringStream` doit écrire `class StringStream(Stream)`.
- `isinstance(s, Stream)` renvoie `True`.
- On peut **mutualiser du code** dans l'ABC (`__enter__`, `__exit__` ci-dessus).
- Tenter d'instancier une sous-classe qui oublie une méthode abstraite échoue **immédiatement**   avec une erreur claire.

In [ ]:
# Oubli d'une méthode → erreur à l'instanciation
class Oubli(Stream):
    def read(self) -> str:
        return "x"

try:
    Oubli()
except TypeError as e:
    print("refusé :", e)

---

## 3. Protocol — typage structurel

Un **Protocol** (PEP 544, Python 3.8+) définit un contrat par **forme** : toute classe qui possède les bons attributs/méthodes est automatiquement compatible, **sans héritage**. C'est le duck typing **vérifié statiquement** (par mypy, pyright).


In [ ]:
from typing import Protocol

class Lisible(Protocol):
    def read(self) -> str: ...

def consomme(source: Lisible) -> int:
    return len(source.read())

class Fichier:  # ne hérite de rien
    def read(self) -> str:
        return "contenu fichier"

class Socket:   # ne hérite de rien non plus
    def read(self) -> str:
        return "contenu socket"

print(consomme(Fichier()))
print(consomme(Socket()))

**Aucune** des deux classes n'hérite de `Lisible`. mypy/pyright vérifient la compatibilité structurellement. C'est plus souple et plus Pythonique : on décrit une capacité, pas une généalogie.

### Syntaxe générique moderne

Avec la syntaxe Python 3.12+, on peut écrire des protocols génériques sans `TypeVar` :

In [ ]:
from typing import Protocol

class Container[T](Protocol):
    def get(self, key: str) -> T | None: ...
    def set(self, key: str, value: T) -> None: ...

class DictBackend:
    def __init__(self) -> None:
        self._d: dict[str, str] = {}
    def get(self, key: str) -> str | None:
        return self._d.get(key)
    def set(self, key: str, value: str) -> None:
        self._d[key] = value

def loader(c: Container[str]) -> None:
    c.set("a", "1")
    print(c.get("a"))

loader(DictBackend())

---

## 4. `@runtime_checkable` — vérification à l'exécution

Par défaut, un `Protocol` n'est **pas** utilisable avec `isinstance()`. Pour l'activer, on ajoute `@runtime_checkable`.

**Attention :** la vérification runtime **ne regarde que la présence** des méthodes, pas leur signature. Elle est donc plus faible que ce que mypy vérifie.


In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Sized(Protocol):
    def __len__(self) -> int: ...

print(isinstance([1, 2, 3], Sized))
print(isinstance("abc", Sized))
print(isinstance(42, Sized))

### Piège : signature non vérifiée à l'exécution

In [ ]:
@runtime_checkable
class Quack(Protocol):
    def quack(self, volume: int) -> str: ...

class MauvaisCanard:
    def quack(self):         # pas d'argument !
        return "coin"

# isinstance dit OUI même si la signature est incompatible
print(isinstance(MauvaisCanard(), Quack))
# L'erreur surviendra à l'appel effectif

---

## 5. ABC + Protocol combinés

Il est parfaitement légal d'avoir **un Protocol pour la signature** et **une ABC pour mutualiser du code**. La bibliothèque standard le fait : `collections.abc.Iterable` est à la fois une ABC et compatible avec les Protocols via `__iter__`.

In [ ]:
from abc import ABC
from typing import Protocol

class Closable(Protocol):
    def close(self) -> None: ...

class Connexion(ABC):
    def __enter__(self): return self
    def __exit__(self, *exc): self.close()
    def close(self) -> None: ...  # laissé à la sous-classe

# Un client peut accepter n'importe quel Closable, y compris des Connexion,
# mais les développeurs de sous-classes de Connexion bénéficient de __enter__/__exit__.


---

## 6. Cas d'usage de chaque outil

### Quand choisir une **ABC**

- Vous écrivez une bibliothèque dont les **utilisateurs doivent hériter** de vos classes.
- Vous voulez **factoriser** du code concret (template method, méthodes de confort).
- Vous voulez que `isinstance()` et `issubclass()` soient **fiables** et obligatoires.
- Vous avez besoin de `@abstractmethod` pour garantir qu'une méthode est implémentée.

Exemples stdlib : `collections.abc.MutableMapping`, `io.IOBase`, `numbers.Number`.

### Quand choisir un **Protocol**

- Vous écrivez une fonction/classe qui **consomme** des objets venant de l'extérieur, parfois   tiers, que vous ne pouvez pas modifier.
- Vous voulez rester **fidèle au duck typing** mais avec l'aide de mypy/pyright.
- Vous décrivez une capacité **ponctuelle** (« quelque chose qui a `.read()` »).
- Vous voulez éviter de forcer vos utilisateurs à hériter d'une classe de votre bibliothèque.

Exemples : `typing.SupportsIndex`, `typing.SupportsFloat`, `os.PathLike`.

---

## 7. Arbre de décision

```
Besoin de décrire un contrat ?
├─ Les objets qui devront le respecter sont déjà écrits (stdlib, tiers) ? → Protocol
├─ Je veux factoriser du code commun (mixin, template method) ? → ABC
├─ Je veux forcer mes utilisateurs à hériter pour avoir accès à ma logique ? → ABC
├─ Je décris une capacité simple et locale ? → Protocol
└─ Je veux les deux ? → ABC **et** Protocol en parallèle
```


---

## 8. Synthèse

| Critère | ABC | Protocol |
|---|---|---|
| Mécanisme | héritage nominal | typage structurel |
| Doit hériter ? | oui | non |
| `isinstance` supporté | toujours | seulement si `@runtime_checkable` |
| Mutualisation de code | oui (méthodes concrètes) | non |
| Vérification de signature | oui (abstractmethod) | statique seulement |
| Convient aux objets tiers | non (nécessite `.register`) | oui par nature |
| Typique pour | frameworks (Django, pytest) | glue code, API de lib |


---

## 9. Exercices

### Exercice 1 — ABC `Shape` *(facile)*

Écrire une ABC `Shape` avec `area() -> float` et `perimeter() -> float` abstraites. Implémenter `Rectangle` et `Circle`. Écrire une fonction `total_area(shapes: list[Shape]) -> float`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Abc_vs_protocol", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...
    @abstractmethod
    def perimeter(self) -> float: ...

class Rectangle(Shape):
    def __init__(self, w: float, h: float) -> None:
        self.w, self.h = w, h
    def area(self) -> float: return self.w * self.h
    def perimeter(self) -> float: return 2 * (self.w + self.h)

class Circle(Shape):
    def __init__(self, r: float) -> None:
        self.r = r
    def area(self) -> float: return math.pi * self.r ** 2
    def perimeter(self) -> float: return 2 * math.pi * self.r

def total_area(shapes: list[Shape]) -> float:
    return sum(s.area() for s in shapes)

print(total_area([Rectangle(2, 3), Circle(1)]))
```

</details>


### Exercice 2 — Protocol `SupportsAdd` *(moyen)*

Écrire un `Protocol` générique `SupportsAdd[T]` qui exige `__add__(self, other: T) -> T`. Écrire une fonction `somme(items: list[SupportsAdd]) -> SupportsAdd` et la tester avec des entiers, des chaînes et une classe `Vector(x, y)` qui implémente `__add__`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Abc_vs_protocol", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Protocol
from functools import reduce

class SupportsAdd[T](Protocol):
    def __add__(self: T, other: T) -> T: ...

def somme[T: SupportsAdd](items: list[T]) -> T:
    return reduce(lambda a, b: a + b, items)

class Vector:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y
    def __add__(self, other: "Vector") -> "Vector":
        return Vector(self.x + other.x, self.y + other.y)
    def __repr__(self) -> str:
        return f"Vector({self.x}, {self.y})"

print(somme([1, 2, 3]))
print(somme(["a", "b", "c"]))
print(somme([Vector(1, 2), Vector(3, 4)]))
```

</details>


### Exercice 3 — Remplacer une ABC par un Protocol dans du code existant *(difficile)*

Voici du code qui utilise une ABC pour décrire un cache. Le réécrire en Protocol pour permettre à des backends tiers (Redis, mémoire) de fonctionner **sans hériter**. Conserver une fonction `benchmark(cache: CacheProtocol)` qui fonctionne avec n'importe quel backend.

In [ ]:
# Code de départ (ABC)
from abc import ABC, abstractmethod

class Cache(ABC):
    @abstractmethod
    def get(self, key: str) -> str | None: ...
    @abstractmethod
    def set(self, key: str, value: str) -> None: ...

# Votre réécriture en Protocol :


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Protocol

class CacheProtocol(Protocol):
    def get(self, key: str) -> str | None: ...
    def set(self, key: str, value: str) -> None: ...

class MemCache:
    def __init__(self) -> None:
        self._d: dict[str, str] = {}
    def get(self, key: str) -> str | None:
        return self._d.get(key)
    def set(self, key: str, value: str) -> None:
        self._d[key] = value

class FakeRedis:  # backend tiers, pas question d'hériter
    def __init__(self) -> None:
        self._d: dict[str, str] = {}
    def get(self, key: str) -> str | None:
        return self._d.get(key)
    def set(self, key: str, value: str) -> None:
        self._d[key] = value

def benchmark(cache: CacheProtocol) -> None:
    cache.set("a", "1")
    print(cache.get("a"))

benchmark(MemCache())
benchmark(FakeRedis())  # marche sans hériter
```

</details>


### Exercice 4 — Limites de `@runtime_checkable` *(deep dive)*

Écrire un `Protocol @runtime_checkable` avec une méthode `compute(x: int) -> int` et montrer par un cas concret que `isinstance` passe alors que l'objet n'implémente **pas** la bonne signature (method non-callable, mauvais nombre d'arguments, etc.). Commenter pourquoi c'est un compromis assumé.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Abc_vs_protocol", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class Computer(Protocol):
    def compute(self, x: int) -> int: ...

# Cas 1 : compute n'est pas callable, juste un attribut
class Cas1:
    compute = 42
print(isinstance(Cas1(), Computer))  # True ! seul le nom est vérifié

# Cas 2 : compute sans argument
class Cas2:
    def compute(self): return 0
print(isinstance(Cas2(), Computer))  # True

# Pourquoi ? La vérification runtime utilise hasattr(obj, 'compute'),
# ce qui est volontairement faible pour rester rapide et compatible avec
# le duck typing. La vérification sérieuse doit être faite statiquement
# (mypy/pyright) ou par des tests.
```

</details>


---

## Ressources

- [PEP 544 — Protocols](https://peps.python.org/pep-0544/)
- [PEP 3119 — ABCs](https://peps.python.org/pep-3119/)
- [docs `typing.Protocol`](https://docs.python.org/3/library/typing.html#typing.Protocol)
- [`collections.abc`](https://docs.python.org/3/library/collections.abc.html)
